# Expert-labeled Subset LLM Scoring

This notebook scores the 400 expert-labeled citation instances using the revised anchored LLM scoring prompt.

Main outputs:

- One continuous relevance score in `[0, 1]` for each model and each citation instance.
- Structured justifications: methodology, theory, evidence, dependency, and summary.
- A checkpointed Excel file that can be resumed safely.
- Optional alignment metrics against expert labels: Spearman, MAE, QWK, and fused Cross+LLM metrics.

Expected input file columns:

- `contexts_text` or `contexts`
- `cited_abstract`
- optional: `final_label`
- optional: `cross_score`

In [ ]:
import os
import re
import json
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

## Configuration

This notebook uses the same OpenAI-compatible client instance for all four LLMs through 智增增:

```python
client = OpenAI(api_key=os.getenv("LLM_KEY"), base_url=os.getenv("LLM_URL"))
```

Set these environment variables before running the notebook:

- `LLM_KEY`
- `LLM_URL`

The four candidate models are selected by changing only the `model` parameter in `client.chat.completions.create(...)`.
If the model names in 智增增 are different from the default strings below, modify only the `model` fields in `MODEL_CONFIGS`.


In [ ]:
INPUT_XLSX = "expert_label.xlsx"
OUTPUT_XLSX = "sample/llms_scored.xlsx"

SAVE_EVERY = 10
REQUEST_SLEEP = 0.5
TEMPERATURE = 0.0

PROMPT_VERSION = "anchored_v1_2026_revision"

# Run range. Use this for segmented/resumable runs.
START_POS = 0
END_POS = None  # None means the end of the file.

# Set to None to run all models, or specify a subset, e.g. ["gpt_5_mini", "qwen3_max"]
MODELS_TO_RUN = None

# If False, existing scores are skipped.
OVERWRITE = False

In [ ]:
@dataclass
class ModelConfig:
    key: str
    model: str
    description: str = ""


MODEL_CONFIGS = {
    "gpt_5_mini": ModelConfig(
        key="gpt_5_mini",
        model="gpt-5-mini",
        description="GPT mini candidate through Zhizengzeng",
    ),
    "qwen3_max": ModelConfig(
        key="qwen3_max",
        model="qwen3-max",
        description="Qwen max candidate through Zhizengzeng",
    ),
    "deepseek_chat": ModelConfig(
        key="deepseek_chat",
        model="deepseek-chat",
        description="DeepSeek chat candidate through Zhizengzeng",
    ),
    "gemini_2_5_flash": ModelConfig(
        key="gemini_2_5_flash",
        model="gemini-2.5-flash",
        description="Gemini flash candidate through Zhizengzeng",
    ),
}

if MODELS_TO_RUN is None:
    MODELS_TO_RUN = list(MODEL_CONFIGS.keys())

MODELS_TO_RUN


## Prompt

The prompt keeps LLM output as a continuous score, but aligns the score range with the four-level expert annotation rubric.

In [ ]:
def build_scoring_prompt(
    context: str,
    abstract: str,
) -> str:
    return f"""
You are evaluating the semantic relevance of one citation instance.

Use only the information provided below.
Do not use citation counts, author identities, affiliations, venue prestige,
or any external knowledge.
Do not assume that every citation is important.
Evaluate the citation according to the explicit relationship demonstrated
in the citation context. If the context is vague, generic, or merely lists
references without a clear relationship, assign a low score.

Citation context:
{context}

Cited paper abstract:
{abstract}

Use the following relevance scale:

0.00--0.24:
No clear relevance, misleading citation, or insufficient contextual evidence.

0.25--0.49:
Weak relevance or broad background citation.

0.50--0.74:
Moderate relevance. The cited paper supports a local claim, concept, method,
comparison, empirical result, or related discussion.

0.75--1.00:
Strong or essential relevance. The cited paper is directly used as a method,
dataset, theory, key empirical evidence, or central foundation.

Return strictly valid JSON:

{{
  "score": 0.00,
  "citation_function": "background | method | theory | evidence | comparison | dataset | definition | motivation | unclear",
  "evidence_alignment": "none | weak | partial | direct",
  "reason": "A concise explanation based only on the supplied citation context and cited-paper information."
}}

The score must be a real number between 0 and 1.
Do not include text outside the JSON object.
""".strip()

## Helper functions

In [ ]:
def contexts_to_text(contexts: Any) -> str:
    if contexts is None or (isinstance(contexts, float) and pd.isna(contexts)):
        return ""

    if isinstance(contexts, str):
        text = contexts.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
            return contexts_to_text(parsed)
        except Exception:
            return text

    if isinstance(contexts, list):
        texts = []
        for idx, item in enumerate(contexts, start=1):
            if isinstance(item, dict):
                text = (
                    item.get("text")
                    or item.get("context")
                    or item.get("sentence")
                    or json.dumps(item, ensure_ascii=False)
                )
            else:
                text = str(item)
            text = text.strip()
            if text:
                texts.append(f"[Context {idx}]\n{text}")
        return "\n\n".join(texts)

    return str(contexts).strip()


def safe_json_loads(text: str) -> dict:
    try:
        return json.loads(text)
    except Exception:
        pass

    # Fallback: extract the first JSON object if the model accidentally returns extra text.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError("No valid JSON object found in model output.")


def normalize_score(value: Any) -> float:
    try:
        score = float(value)
    except Exception:
        score = 0.0
    return float(min(max(score, 0.0), 1.0))


def get_client() -> OpenAI:
    api_key = os.getenv("LLM_KEY")
    base_url = os.getenv("LLM_URL")

    if not api_key:
        raise ValueError("Missing API key: environment variable `LLM_KEY` is not set.")
    if not base_url:
        raise ValueError("Missing base URL: environment variable `LLM_URL` is not set.")

    return OpenAI(api_key=api_key, base_url=base_url)


def output_columns(model_key: str) -> list[str]:
    return [
        f"{model_key}_score",
        f"{model_key}_function",
        f"{model_key}_evidence",
        f"{model_key}_reason",
        f"{model_key}_raw",
        f"{model_key}_error",
        f"{model_key}_model",
        f"{model_key}_scored_at",
    ]


In [ ]:
def score_one_instance(
    client: OpenAI,
    model: str,
    context: str,
    cited_abstract: str,
    temperature: float = 0.0,
) -> dict:
    prompt = build_scoring_prompt(context=context, abstract=cited_abstract)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a careful and neutral academic reviewer."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content or ""
    result = safe_json_loads(raw)

    return {
        "score": normalize_score(result.get("score", 0.0)),
        "function": str(result.get("citation_function", "")),
        "evidence": str(result.get("evidence_alignment", "")),
        "reason": str(result.get("reason", "")),
        "raw": raw,
        "error": "",
    }


## Load input data

In [ ]:
def load_or_resume(input_xlsx: str, output_xlsx: str) -> pd.DataFrame:
    if Path(output_xlsx).exists():
        print(f"Resume from existing output: {output_xlsx}")
        df = pd.read_excel(output_xlsx)
    else:
        print(f"Load input: {input_xlsx}")
        df = pd.read_excel(input_xlsx)

    if "citation_context" not in df.columns and "contexts" not in df.columns:
        raise ValueError("Input file must contain `citation_context` or `contexts`.")

    if "cited_abstract" not in df.columns:
        raise ValueError("Input file must contain `cited_abstract`.")

    # Create normalized input fields used by the scoring function.
    if "citation_context" in df.columns:
        df["_llm_context"] = df["citation_context"].apply(contexts_to_text)
    else:
        df["_llm_context"] = df["contexts"].apply(contexts_to_text)

    df["_llm_cited_abstract"] = df["cited_abstract"].fillna("").astype(str)

    # If expert labels are available, normalize them for later evaluation.
    if "final_label" in df.columns:
        labels = pd.to_numeric(df["final_label"], errors="coerce")
        min_label = labels.min()
        max_label = labels.max()
        if pd.notna(min_label) and pd.notna(max_label):
            if min_label >= 0 and max_label <= 3:
                df["expert_score"] = labels / 3.0
                df["expert_label_display"] = labels + 1
            elif min_label >= 1 and max_label <= 4:
                df["expert_score"] = (labels - 1) / 3.0
                df["expert_label_display"] = labels
            else:
                print(f"Warning: unexpected final_label range: {min_label}--{max_label}")

    return df


df = load_or_resume(INPUT_XLSX, OUTPUT_XLSX)
print(df.shape)
df.head(2)

## Run LLM scoring

This loop is resumable. Existing scores are skipped unless `OVERWRITE=True`.

In [ ]:
def ensure_model_columns(df: pd.DataFrame, model_key: str) -> pd.DataFrame:
    for col in output_columns(model_key):
        if col not in df.columns:
            df[col] = None
    return df


def should_skip(row: pd.Series, model_key: str, overwrite: bool = False) -> bool:
    if overwrite:
        return False
    score_col = f"{model_key}_score"
    error_col = f"{model_key}_error"
    has_score = pd.notna(row.get(score_col))
    has_error = pd.notna(row.get(error_col)) and str(row.get(error_col)).strip() != ""
    return bool(has_score and not has_error)


def save_checkpoint(df: pd.DataFrame, output_xlsx: str):
    Path(output_xlsx).parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(output_xlsx, index=False)


def score_dataframe(
    df: pd.DataFrame,
    model_keys: list[str],
    output_xlsx: str,
    start_pos: int = 0,
    end_pos: int | None = None,
    overwrite: bool = False,
) -> pd.DataFrame:
    total_rows = len(df)
    if end_pos is None:
        end_pos = total_rows

    start_pos = max(0, int(start_pos))
    end_pos = min(int(end_pos), total_rows)

    if start_pos >= end_pos:
        raise ValueError(f"Invalid range: start_pos={start_pos}, end_pos={end_pos}, total_rows={total_rows}")

    for model_key in model_keys:
        if model_key not in MODEL_CONFIGS:
            raise KeyError(f"Unknown model key: {model_key}")
        df = ensure_model_columns(df, model_key)

    # 智增增：四个模型使用同一个 OpenAI-compatible client，只切换 model 参数。
    client = get_client()

    for model_key in model_keys:
        config = MODEL_CONFIGS[model_key]

        positions = list(range(start_pos, end_pos))
        pbar = tqdm(positions, desc=f"Scoring with {model_key}", total=len(positions))

        processed = 0
        skipped = 0
        failed = 0

        for pos in pbar:
            idx = df.index[pos]
            row = df.loc[idx]

            if should_skip(row, model_key=model_key, overwrite=overwrite):
                skipped += 1
                continue

            context = str(row.get("_llm_context", "") or "").strip()
            cited_abstract = str(row.get("_llm_cited_abstract", "") or "").strip()

            if not context or not cited_abstract:
                df.at[idx, f"{model_key}_score"] = np.nan
                df.at[idx, f"{model_key}_error"] = "missing_context_or_cited_abstract"
                failed += 1
                continue

            citing_id = row.get("citing_corpus_id", "")
            cited_id = row.get("cited_corpus_id", "")
            pbar.set_postfix_str(f"row={pos + 1}/{total_rows}, {citing_id}->{cited_id}")

            try:
                result = score_one_instance(
                    client=client,
                    model=config.model,
                    context=context,
                    cited_abstract=cited_abstract,
                    temperature=TEMPERATURE,
                )

                df.at[idx, f"{model_key}_score"] = result["score"]
                df.at[idx, f"{model_key}_function"] = result["function"]
                df.at[idx, f"{model_key}_evidence"] = result["evidence"]
                df.at[idx, f"{model_key}_reason"] = result["reason"]
                df.at[idx, f"{model_key}_raw"] = result["raw"]
                df.at[idx, f"{model_key}_error"] = ""
                df.at[idx, f"{model_key}_model"] = config.model
                df.at[idx, f"{model_key}_scored_at"] = datetime.now().isoformat(timespec="seconds")

                processed += 1

            except Exception as e:
                df.at[idx, f"{model_key}_score"] = np.nan
                df.at[idx, f"{model_key}_error"] = repr(e)
                df.at[idx, f"{model_key}_model"] = config.model
                df.at[idx, f"{model_key}_scored_at"] = datetime.now().isoformat(timespec="seconds")
                failed += 1

            if REQUEST_SLEEP:
                time.sleep(REQUEST_SLEEP)

            if (processed + failed) % SAVE_EVERY == 0:
                save_checkpoint(df, output_xlsx)

        save_checkpoint(df, output_xlsx)
        print(
            f"{model_key}: processed={processed}, skipped={skipped}, failed={failed}, saved={output_xlsx}"
        )

    return df


In [ ]:
df = score_dataframe(
    df=df,
    model_keys=MODELS_TO_RUN,
    output_xlsx=OUTPUT_XLSX,
    start_pos=START_POS,
    end_pos=END_POS,
    overwrite=OVERWRITE,
)

print(f"Saved to: {OUTPUT_XLSX}")
df.head(2)